# Python Basics

## Variable Assignment

A variable can be created and assigned a value using the assignment operator `=`. The value of the variable can be updated by reassigning it with a new value.

In [ ]:
x = 10
x = x + 5
print(x) 

## Defining Functions

A function is a block of code that performs a specific task. It can take inputs, called parameters, and can return an output. Functions are defined using the `def` keyword.

In [ ]:
def add_numbers(a, b):
    return a + b

print(add_numbers(3, 4))

## Lists and Dictionaries

Lists and dictionaries can be used to store multiple values or variables.
* A list is an ordered collection of items, defined using square brackets `[]`. Items in a list can be accessed by their index, starting from 0.
* A dictionary is an unordered collection of key-value pairs, defined using curly braces `{}`. Values can be accessed using their corresponding keys.

In [ ]:
a = 5
b = 10
c = 15
my_list = [a, b, c]
for item in my_list:
    print(item)

In [ ]:
name = "Alice"
age = 30
person = {
    "name": name,
    "age": age
}
print("Name:", person["name"])
print("Age:", person["age"])

# Functions as Values?

Functions can actually be treated as values in Python, so many of the same operations that can be performed on other values can also be performed on functions.

For example, functions can be assigned to variables:

In [ ]:
def add_numbers(a, b):
    return a + b

add = add_numbers
result = add(3, 4)
print(result)

Or can be assigned to a dictionary:

In [ ]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

operations = {
    "+": add,
    "-": subtract
}

def operate(sign, operands):
    result = operations[sign](operands[0], operands[1])
    print(operands[0], sign, operands[1], "=", result)
    return result

operate("+", [10, 5])
operate("-", [10, 5])

Or even passed as arguments to other functions:

In [ ]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def operate(operation, a, b):
    operation_name = operation.__name__
    result = operation(a, b)
    print(f"{operation_name}({a}, {b}) = {result}")
    return result

operate(add, 10, 5)
operate(subtract, 10, 5)

Or returned as values from other functions:

In [ ]:
def create_operation(sign):
    def _add(a, b):
        result = a + b
        print(a, sign, b, "=", result)
        return result

    def _subtract(a, b):
        result = a - b
        print(a, sign, b, "=", result)
        return result

    if sign == "+":
        return _add
    elif sign == "-":
        return _subtract

    def unknown_operation(a, b):
        raise ValueError(f"Unknown operation: {sign}")
    return unknown_operation

create_operation("+")(10, 5)
create_operation("-")(10, 5)

A function that takes another function, or returns a function, is called a **higher-order function**. This unlocks many neat techniques, such as **decorators**.

# Decorators

We see from the above that I add a print statement to almost all of my functions so I can see which functions are being called and when; and what their inputs and outputs are. This is a common pattern:

```python
def my_function(x):
    print(f"my_function called with x = {x}")
    result = x + 1
    print(f"my_function returning {result}")
    return result
```

The actual logic of the function is only a single line `result = x + 1`, but the print statements are cluttering up the function.

What if there is a way to add this logging functionality to **any** function, without having to modify the function itself?

## Decoupling

First, we can decouple (separate) the logging functionality from the actual logic of the function

In [ ]:
def increment(x):
    return x + 1

def increment_with_print(x):
    func_name = "increment"
    print(f"{func_name} called with x = {x}")
    result = increment(x)
    print(f"{func_name} returned {result}")
    return result - 1

increment_with_print(10)

## Creating a Higher-Order Function

It's not too difficult to see that, regardless of the actual logic of the function, the logging functionality is always the same, like so:

```python
def func_with_print(x):
    func_name = func.__name__
    print(f"{func_name} called with x = {x}")
    result = func(x)
    print(f"{func_name} returned {result}")
    return result
```

And `func` can be any function that takes a single argument and returns a value.

From the lesson above, we know that functions can be passed as arguments to other functions, and we can also construct a function inside another function and return it.

In [ ]:
def with_print(func):
    def func_with_print(x):
        func_name = func.__name__
        print(f"{func_name} called with x = {x}")
        result = func(x)
        print(f"{func_name} returned {result}")
        return result
    return func_with_print

increment_with_print = with_print(increment)
increment_with_print(10)

And we don't even have to create a new function name! We can actually replace the original function with the new function that has the logging functionality added to it.

In [ ]:
def decrement(x):
    return x - 1

decrement = with_print(decrement)
decrement(10)

## The `@` Syntax

The above is a simple example of a **decorator**. A decorator is a function that takes another function and extends its behavior without explicitly modifying it.

Python has a special syntax for decorators, using the `@` symbol.

```python
@decorator
def my_function(x):
    # Do something with x
    pass
```

In [ ]:
@with_print
def square(x):
    return x * x

square(10)

# Advanced Examples

## Multiple Arguments and Keyword Arguments

In the above example, the decorator only works for functions that take a single argument. But what if we want to decorate a function that takes multiple arguments, or keyword arguments?

Python has a special syntax for this, using `*args` and `**kwargs`. These are used to pass a variable number of arguments to a function.

```python
def decorator(func):
    def func_with_decorator(*args, **kwargs):
        result = func(*args, **kwargs)
        return result
    return func_with_decorator
```

In [ ]:
def print_it(func):
    def func_with_print(*args, **kwargs):
        func_name = func.__name__
        print(f"{func_name} called with args = {args}, kwargs = {kwargs}")
        result = func(*args, **kwargs)
        print(f"{func_name} returned {result}")
        return result
    return func_with_print

@print_it
def add(a, b):
    return a + b

@print_it
def power(base, exponent):
    return base ** exponent

add(3, 4)
power(2, 3)
power(base=4, exponent=6)

## Timing and Caching

Other common use cases for decorators are:
* Timing: measuring how long a function takes to run
* Caching: storing the results of expensive function calls and returning the cached result when the same inputs occur again

Let's see these in action with the **Fibonacci sequence**. The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding ones, starting from 0 and 1. The sequence goes like this:

```
0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, etc.
```

And can be defined recursively as:

```python
def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)
```

To time the execution of this function, we want a decorator that does something like this (using single argument for simplicity):

```python
from time import time

def func_with_time(n):
    start_time = time.time()
    result = func(n)
    end_time = time.time()
    if elapsed_time > 0.5: print(f"{func.__name__}({n}) took {end_time - start_time} seconds to run")
    return result
```

In [ ]:
from time import time

def time_it(func):
    def func_with_time(n):
        start_time = time()
        result = func(n)
        end_time = time()
        elapsed_time = end_time - start_time
        if elapsed_time > 0.5: print(f"{func.__name__}({n}) took {elapsed_time:.6f} seconds to execute.")
        return result
    return func_with_time

@time_it
def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(35)

Caching is the process of storing the results of expensive function calls in a cache (a temporary storage area) so that the next time the function is called with the same inputs, the cached result can be returned instead of recomputing it.

A dictionary is an example of a simple cache, where the keys are the inputs to the function and the values are the results of the function calls.

```python
cache = {}

def func_with_cache(n):
    # If found, return the cached result
    if n in cache:
        return cache[n]
    # Otherwise, compute the result and store it in the cache
    result = func(n)
    cache[n] = result
    return result
```

In [ ]:
cache = {}

def cache_it(func):
    def func_with_cache(n):
        # If found, return the cached result
        if n in cache:
            return cache[n]
        # Otherwise, compute the result and store it in the cache
        result = func(n)
        cache[n] = result
        return result
    return func_with_cache

@time_it
@cache_it
def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(35)